In [14]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "mdlr-504420"
DATASET_ID = "telecom_analytics"
TABLE_ID    = "churn_analysis"

client = bigquery.Client.from_service_account_json("../service_account.json", project=PROJECT_ID)
print("✅ BigQuery client initialized.")

✅ BigQuery client initialized.


In [15]:
create_dataset_query = f"""
CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}`
OPTIONS (location = 'US');
"""
client.query(create_dataset_query).result()
print(f"✅ Dataset '{DATASET_ID}' ready.")

✅ Dataset 'telecom_analytics' ready.


In [16]:
create_table_query = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` AS
SELECT
  customerID,
  tenure,
  MonthlyCharges,
  TotalCharges,
  Contract,
  InternetService,
  OnlineSecurity,
  TechSupport,
  StreamingTV,
  PaymentMethod,
  CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END AS churn_label
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE TotalCharges IS NOT NULL;
"""
client.query(create_table_query).result()
print("✅ Training table created.")

BadRequest: 400 Unrecognized name: Churn at [14:13]; reason: invalidQuery, location: query, message: Unrecognized name: Churn at [14:13]

Location: us-central1
Job ID: 5a3033a0-7d55-457b-a76d-8bf08dddfd40


In [ ]:
train_model_query = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.churn_model`
OPTIONS (
  model_type        = 'LOGISTIC_REG',
  input_label_cols  = ['churn_label'],
  auto_class_weights = TRUE,
  data_split_method = 'AUTO_SPLIT'
) AS
SELECT
  tenure,
  MonthlyCharges,
  TotalCharges,
  Contract,
  InternetService,
  OnlineSecurity,
  TechSupport,
  StreamingTV,
  PaymentMethod,
  churn_label
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`;
"""
client.query(train_model_query).result()
print("✅ Model trained successfully.")

✅ Model trained successfully.


In [ ]:
eval_query = f"""
SELECT
  precision,
  recall,
  accuracy,
  f1_score,
  roc_auc
FROM ML.EVALUATE(
  MODEL `{PROJECT_ID}.{DATASET_ID}.churn_model`,
  (SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`)
);
"""
eval_df = client.query(eval_query).to_dataframe()
print("📊 Model Evaluation Results:")
display(eval_df)

/Users/michaelmeza/pyspark-telecom-pipeline/venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


📊 Model Evaluation Results:


,precision,recall,accuracy,f1_score,roc_auc
0,0.502875,0.795613,0.736633,0.616245,0.837084


In [ ]:
predict_query = f"""
SELECT
  customerID,
  predicted_churn_label,
  predicted_churn_label_probs
FROM ML.PREDICT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.churn_model`,
  (
    SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    LIMIT 100
  )
);
"""
predictions_df = client.query(predict_query).to_dataframe()
print("🔮 Sample Predictions (top 10):")
display(predictions_df.head(10))

/Users/michaelmeza/pyspark-telecom-pipeline/venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


🔮 Sample Predictions (top 10):


,customerID,predicted_churn_label,predicted_churn_label_probs
0,2967-MXRAV,0,"[{'label': 1, 'prob': 0.1966222814409087}, {'l..."
1,2967-MXRAV,0,"[{'label': 1, 'prob': 0.1966222814409087}, {'l..."
2,8992-CEUEN,0,"[{'label': 1, 'prob': 0.41236386377912637}, {'..."
3,8992-CEUEN,0,"[{'label': 1, 'prob': 0.41236386377912637}, {'..."
4,9318-NKNFC,0,"[{'label': 1, 'prob': 0.3136359920995942}, {'l..."
5,9318-NKNFC,0,"[{'label': 1, 'prob': 0.3136359920995942}, {'l..."
6,9975-SKRNR,0,"[{'label': 1, 'prob': 0.3136868389750759}, {'l..."
7,9975-SKRNR,0,"[{'label': 1, 'prob': 0.3136868389750759}, {'l..."
8,1423-BMPBQ,0,"[{'label': 1, 'prob': 0.3137885461510966}, {'l..."
9,1423-BMPBQ,0,"[{'label': 1, 'prob': 0.3137885461510966}, {'l..."


In [ ]:
importance_query = f"""
SELECT
  processed_input AS feature,
  weight
FROM ML.WEIGHTS(
  MODEL `{PROJECT_ID}.{DATASET_ID}.churn_model`
)
ORDER BY ABS(weight) DESC;
"""

In [ ]:
export_query = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.dm_churn_ml` AS
SELECT
  c.customerID,
  c.tenure,
  c.MonthlyCharges,
  c.Contract,
  c.churn_label,
  p.predicted_churn_label,
  (SELECT prob.prob FROM UNNEST(p.predicted_churn_label_probs) prob
   WHERE prob.label = 1) AS churn_probability
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` c
JOIN ML.PREDICT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.churn_model`,
  (SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`)
) p
USING (customerID)
ORDER BY churn_probability DESC;
"""
client.query(export_query).result()
print("✅ dm_churn_ml table created and ready.")

✅ dm_churn_ml table created and ready.
